no pitch no roll

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime
from subprocess import run
from PIL import Image
import piexif
import shutil

# ============================================================
# ===================== ROOT INPUT ===========================
# ============================================================

FRAMES_DIR_ROOT = r"E:\04.Pak_ival\Dataset\merdeka\4.frame_360_split\3_1301"
METADATA_ROOT   = r"E:\04.Pak_ival\Dataset\merdeka\1.360_raw\output"  # ← hasil STEP 1
VIDEO_INFO_TXT  = r"E:\04.Pak_ival\Dataset\merdeka\3.frame_360\3\summary_txt_fps3.txt"
OUTPUT_ROOT     = r"E:\04.Pak_ival\Dataset\merdeka\5.camera_geotag_smooth"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ============================================================
# ================= BACA DURASI TIAP VIDEO ===================
# ============================================================

def load_video_durations(txt_path):
    video_data = {}
    with open(txt_path, "r") as f:
        lines = f.readlines()

    current_name = None
    for line in lines:
        line = line.strip()
        if line.startswith("Nama"):
            current_name = line.split(":")[1].strip().replace(".mp4", "")
        elif line.startswith("Durasi") and current_name is not None:
            dur = float(line.split(":")[1].strip().split()[0])
            video_data[current_name] = dur
            current_name = None

    return video_data


print("\n=== LOAD VIDEO DURATIONS FROM TXT ===")
video_durations = load_video_durations(VIDEO_INFO_TXT)
print(f"Durasi ditemukan untuk {len(video_durations)} video\n")

# ============================================================
# ===================== HELPER FUNCTIONS =====================
# ============================================================

def deg_to_dms_rational(deg_float):
    deg = int(deg_float)
    min_float = abs(deg_float - deg) * 60
    minutes = int(min_float)
    sec = round((min_float - minutes) * 60 * 100)
    return ((abs(deg), 1), (minutes, 1), (sec, 100))

# ============================================================
# ======================= MAIN LOOP ==========================
# ============================================================

video_folders = sorted([
    d for d in os.listdir(FRAMES_DIR_ROOT)
    if os.path.isdir(os.path.join(FRAMES_DIR_ROOT, d))
])

print("=== FOUND VIDEO FOLDERS ===")
for v in video_folders:
    print(" -", v)
print()

for video_id in video_folders:

    print("\n==========================================================")
    print(f"======= MEMPROSES VIDEO: {video_id} =======")
    print("==========================================================")

    FRAMES_DIR = os.path.join(FRAMES_DIR_ROOT, video_id)
    CSV_PATH   = os.path.join(METADATA_ROOT, f"{video_id}.csv")

    if not os.path.exists(CSV_PATH):
        print(f"CSV {video_id}.csv tidak ditemukan, SKIP!")
        continue

    VIDEO_DURATION = video_durations.get(video_id)
    if VIDEO_DURATION is None:
        print(f"Durasi video {video_id} tidak ditemukan, SKIP!")
        continue

    print(f"Durasi video: {VIDEO_DURATION} detik")
    print(f"CSV metadata ditemukan: {CSV_PATH}")

    # ========================================================
    # =============== PART 1 — CSV MAPPING ===================
    # ========================================================

    gps_df = pd.read_csv(CSV_PATH).sort_values("t_rel_sec").reset_index(drop=True)
    t_min, t_max = gps_df["t_rel_sec"].min(), gps_df["t_rel_sec"].max()

    img_files = sorted([
        f for f in os.listdir(FRAMES_DIR)
    if re.match(r"^GS\d+_image_\d+_\d+\.jpg", f)
    ])

    grouped = {}
    for fname in img_files:
        m = re.match(r"^GS\d+_image_(\d+)_\d+\.jpg", fname)
        if m:
            gid = int(m.group(1))
            grouped.setdefault(gid, []).append(fname)

    group_ids = sorted(grouped.keys())
    usable = min(len(group_ids), len(gps_df))
    group_ids = group_ids[:usable]

    timestamps = np.linspace(0, VIDEO_DURATION, len(group_ids))

    cols_to_interp = [
        "lat","lon","alt","spd_3d","spd_2d","yaw",
        "gyro_x","gyro_y","gyro_z",
        "accel_x","accel_y","accel_z","temp"
    ]

    mapping_rows = []

    for gid, t_vid in zip(group_ids, timestamps):
        t_clipped = np.clip(t_vid, t_min, t_max)

        interpolated = {
            col: np.interp(t_clipped, gps_df["t_rel_sec"], gps_df[col])
            if col in gps_df.columns else np.nan
            for col in cols_to_interp
        }

        base_yaw = interpolated["yaw"]

        subfiles = sorted(
            grouped[gid],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )

        n_sub = len(subfiles)

        for i, subf in enumerate(subfiles):
            step = 360.0 / n_sub
            sub_yaw = (base_yaw + i * step + 360) % 360

            row = {
                "image_name": subf,
                "timestamp_video": t_vid,
                **interpolated
            }
            row["yaw"] = sub_yaw
            mapping_rows.append(row)

    mapping_df = pd.DataFrame(mapping_rows)

    # ========================================================
    # =============== PART 2 — EXIF + XMP ====================
    # ========================================================

    OUT_IMG_DIR = os.path.join(OUTPUT_ROOT, video_id)
    os.makedirs(OUT_IMG_DIR, exist_ok=True)

    for _, row in mapping_df.iterrows():

        src = os.path.join(FRAMES_DIR, row["image_name"])
        dst = os.path.join(OUT_IMG_DIR, row["image_name"])

        img = Image.open(src)
        exif = {"0th": {}, "Exif": {}, "GPS": {}, "1st": {}, "thumbnail": None}

        lat, lon, alt = row["lat"], row["lon"], row["alt"]

        exif["GPS"][piexif.GPSIFD.GPSLatitudeRef]  = "S" if lat < 0 else "N"
        exif["GPS"][piexif.GPSIFD.GPSLatitude]     = deg_to_dms_rational(lat)
        exif["GPS"][piexif.GPSIFD.GPSLongitudeRef] = "W" if lon < 0 else "E"
        exif["GPS"][piexif.GPSIFD.GPSLongitude]    = deg_to_dms_rational(lon)
        exif["GPS"][piexif.GPSIFD.GPSAltitude]     = (int(alt * 100), 100)

        yaw = float(row["yaw"])

        exif["GPS"][piexif.GPSIFD.GPSImgDirectionRef] = "T"
        exif["GPS"][piexif.GPSIFD.GPSImgDirection] = (int(yaw * 100), 100)

        meta_json = {
            "yaw_deg": yaw
        }

        exif["0th"][piexif.ImageIFD.ImageDescription] = json.dumps(meta_json).encode()
        img.save(dst, exif=piexif.dump(exif))

        run([
            "exiftool", "-overwrite_original",
            f"-XMP:PoseYawDegrees={yaw}",
            dst
        ])

        print(f"{video_id} — {row['image_name']} → OK")

    print(f"\nSELESAI VIDEO: {video_id}\n")
    # ============================================================
    # =============== PART 3 — AUTO SPLIT FILE ===================
    # ============================================================

    print("\n=== PART 3: SPLIT KE FOLDER BERDASARKAN INDEX ===")

    base_dir = OUT_IMG_DIR  # gunakan folder output dari video ini

    for filename in os.listdir(base_dir):
        if "_" in filename:
            try:
                index = filename.split("_")[-1].replace(".jpg", "")
                folder_name = f"folder_{index}"
                folder_path = os.path.join(base_dir, folder_name)
                os.makedirs(folder_path, exist_ok=True)

                src = os.path.join(base_dir, filename)
                dst = os.path.join(folder_path, filename)
                shutil.move(src, dst)

                print(f"Moved {filename} -> {folder_name}")

            except Exception as e:
                print(f"Skipping {filename}: {e}")

    print(f" SPLIT SELESAI untuk {video_id}\n")

print("\n==================== ALL VIDEOS DONE ====================\n")



=== LOAD VIDEO DURATIONS FROM TXT ===
Durasi ditemukan untuk 6 video

=== FOUND VIDEO FOLDERS ===
 - GS014419


======= MEMPROSES VIDEO: GS014419 =======
Durasi video: 519.05 detik
CSV metadata ditemukan: E:\04.Pak_ival\Dataset\merdeka\1.360_raw\output\GS014419.csv
GS014419 — GS014419_image_0001_0.jpg → OK
GS014419 — GS014419_image_0001_1.jpg → OK
GS014419 — GS014419_image_0001_2.jpg → OK
GS014419 — GS014419_image_0001_3.jpg → OK
GS014419 — GS014419_image_0001_4.jpg → OK
GS014419 — GS014419_image_0001_5.jpg → OK
GS014419 — GS014419_image_0001_6.jpg → OK
GS014419 — GS014419_image_0001_7.jpg → OK
GS014419 — GS014419_image_0002_0.jpg → OK
GS014419 — GS014419_image_0002_1.jpg → OK
GS014419 — GS014419_image_0002_2.jpg → OK
GS014419 — GS014419_image_0002_3.jpg → OK
GS014419 — GS014419_image_0002_4.jpg → OK
GS014419 — GS014419_image_0002_5.jpg → OK
GS014419 — GS014419_image_0002_6.jpg → OK
GS014419 — GS014419_image_0002_7.jpg → OK
GS014419 — GS014419_image_0003_0.jpg → OK
GS014419 — GS014419

pitch and roll

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime
from subprocess import run
from PIL import Image
import piexif
import shutil

# ============================================================
# ===================== ROOT INPUT ===========================
# ============================================================

# FRAMES_DIR_ROOT = r"D:\2025.Gaussian_Splatting\dataset\sumurbandung\10nov25\4.frame_360_split\3_1300\part2"
FRAMES_DIR_ROOT = r"D:\metadata\360\GAGAL\4.frame_360_split\3_1300"
METADATA_ROOT   = r"D:\metadata\360\GAGAL\1.360_raw\output"
# VIDEO_INFO_TXT  = os.path.join(METADATA_ROOT, "summary_txt_fps3.txt")
# VIDEO_INFO_TXT = r"D:\2025.Gaussian_Splatting\dataset\sumurbandung\10nov25\3.frame_360\3\summary_txt_fps3.txt"
VIDEO_INFO_TXT = r"D:\metadata\360\GAGAL\3.frame_360\3\summary_txt_fps3.txt"
OUTPUT_ROOT = r"D:\metadata\360\GAGAL\5.camera_geotag"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ============================================================
# ================= BACA DURASI TIAP VIDEO ===================
# ============================================================

def load_video_durations(txt_path):
    video_data = {}
    with open(txt_path, "r") as f:
        lines = f.readlines()

    current_name = None
    for line in lines:
        line = line.strip()
        if line.startswith("Nama"):
            current_name = line.split(":")[1].strip().replace(".mp4", "")
        elif line.startswith("Durasi") and current_name is not None:
            dur = float(line.split(":")[1].strip().split()[0])
            video_data[current_name] = dur
            current_name = None

    return video_data

print("\n=== LOAD VIDEO DURATIONS FROM TXT ===")
video_durations = load_video_durations(VIDEO_INFO_TXT)
print(f"Durasi ditemukan untuk {len(video_durations)} video\n")

# ============================================================
# ===================== HELPER FUNCTIONS =====================
# ============================================================

def deg_to_dms_rational(deg_float):
    deg = int(deg_float)
    min_float = abs(deg_float - deg) * 60
    minutes = int(min_float)
    sec = round((min_float - minutes) * 60 * 100)
    return ((abs(deg), 1), (minutes, 1), (sec, 100))


# ============================================================
# ======================= MAIN LOOP ==========================
# ============================================================

video_folders = sorted([
    d for d in os.listdir(FRAMES_DIR_ROOT)
    if os.path.isdir(os.path.join(FRAMES_DIR_ROOT, d))
])

print("=== FOUND VIDEO FOLDERS ===")
for v in video_folders:
    print(" -", v)
print()

for video_id in video_folders:

    print("\n==========================================================")
    print(f"======= MEMPROSES VIDEO: {video_id} =======")
    print("==========================================================")

    FRAMES_DIR = os.path.join(FRAMES_DIR_ROOT, video_id)
    CSV_PATH   = os.path.join(METADATA_ROOT, f"{video_id}.csv")

    if not os.path.exists(CSV_PATH):
        print(f"CSV {video_id}.csv tidak ditemukan, SKIP video ini!")
        continue

    VIDEO_DURATION = video_durations.get(video_id, None)
    if VIDEO_DURATION is None:
        print(f"Durasi video {video_id} tidak ditemukan di TXT, SKIP!")
        continue

    print(f"Durasi video: {VIDEO_DURATION} detik")
    print(f"CSV metadata ditemukan: {CSV_PATH}")

    # ============================================================
    # =============== PART 1 — BUAT CSV MAPPING ==================
    # ============================================================

    print("\n=== PART 1: GENERATE CSV MAPPING ===")

    gps_df = pd.read_csv(CSV_PATH)
    gps_df = gps_df.sort_values("t_rel_sec").reset_index(drop=True)

    t_min, t_max = gps_df["t_rel_sec"].min(), gps_df["t_rel_sec"].max()

    img_files = sorted([
        f for f in os.listdir(FRAMES_DIR)
        # if re.match(r"image_\d+_\d+\.jpg", f)
        if re.match(r"^GS\d+_image_\d+_\d+\.jpg", f)
    ])

    if len(img_files) == 0:
        print("Tidak ditemukan file GSXXXX_image_xxx_x.jpg → SKIP video ini!")
        continue

    grouped = {}
    for fname in img_files:
        # m = re.match(r"image_(\d+)_\d+\.jpg", fname)
        m = re.match(r"^GS\d+_image_(\d+)_\d+\.jpg", fname)
        if m:
            gid = int(m.group(1))
            grouped.setdefault(gid, []).append(fname)

    group_ids = sorted(grouped.keys())
    n_frames= len(group_ids)
    n_meta = len(gps_df)

    print(f"jumlah grup frame:{n_frames}")
    print(f"jumlah metadata row: {n_meta}")

    usable = min(n_frames, n_meta)

    if n_frames != n_meta:
        print("jumlah metadata tidak sama")
        print(f"yang di proses hanya: {usable}")
        print(f"sisa frame: {n_frames - usable}")
        print(f"sisa_metadata : {n_meta - usable}")

    group_ids = group_ids[:usable]
    # timestamps = np.linspace[0,VIDEO_DURATION, usable]

    timestamps = np.linspace(0, VIDEO_DURATION, len(group_ids))

    cols_to_interp = [
        "lat","lon","alt","spd_3d","spd_2d",
        "yaw","pitch","roll",
        "gyro_x","gyro_y","gyro_z",
        "accel_x","accel_y","accel_z",
        "temp"
    ]

    mapping_rows = []

    for gid, t_vid in zip(group_ids, timestamps):
        t_clipped = np.clip(t_vid, t_min, t_max)

        interpolated = {
            col: np.interp(t_clipped, gps_df["t_rel_sec"], gps_df[col])
            if col in gps_df.columns else np.nan
            for col in cols_to_interp
        }

        base_yaw = interpolated["yaw"]

        subfiles = sorted(
            grouped[gid],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )
        n_sub = len(subfiles)

        for i, subf in enumerate(subfiles):
            step = 360.0 / n_sub
            sub_yaw = (base_yaw + i * step + 360) % 360

            row = {
                "image_name": subf,
                "timestamp_video": t_vid,
                **{c: interpolated[c] for c in cols_to_interp}
            }
            row["yaw"] = sub_yaw
            mapping_rows.append(row)

    mapping_df = pd.DataFrame(mapping_rows)

    out_csv = os.path.join(OUTPUT_ROOT, f"{video_id}_mapping.csv")
    mapping_df.to_csv(out_csv, index=False)

    print(f"CSV mapping selesai → {out_csv}")

    # ============================================================
    # =============== PART 2 — TULIS EXIF ========================
    # ============================================================

    print("\n=== PART 2: EMBED EXIF + XMP ===")

    OUT_IMG_DIR = os.path.join(OUTPUT_ROOT, video_id)
    os.makedirs(OUT_IMG_DIR, exist_ok=True)

    for _, row in mapping_df.iterrows():

        img_name = row["image_name"]
        src_path = os.path.join(FRAMES_DIR, img_name)
        dst_path = os.path.join(OUT_IMG_DIR, img_name)

        if not os.path.exists(src_path):
            print(f" Skip: {src_path} hilang")
            continue

        img = Image.open(src_path)
        exif_dict = {"0th": {}, "Exif": {}, "GPS": {}, "1st": {}, "thumbnail": None}

        lat, lon, alt = row["lat"], row["lon"], row["alt"]

        exif_dict["GPS"][piexif.GPSIFD.GPSLatitudeRef]  = "S" if lat < 0 else "N"
        exif_dict["GPS"][piexif.GPSIFD.GPSLatitude]     = deg_to_dms_rational(lat)
        exif_dict["GPS"][piexif.GPSIFD.GPSLongitudeRef] = "W" if lon < 0 else "E"
        exif_dict["GPS"][piexif.GPSIFD.GPSLongitude]    = deg_to_dms_rational(lon)
        exif_dict["GPS"][piexif.GPSIFD.GPSAltitude]     = (int(alt * 100), 100)

        # datetime
        try:
            dt_obj = datetime.fromisoformat(str(row.get("datetime", "")))
            dt_fmt = dt_obj.strftime("%Y:%m:%d %H:%M:%S")
            exif_dict["0th"][piexif.ImageIFD.DateTime] = dt_fmt
            exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal] = dt_fmt
        except:
            pass

        yaw = float(row["yaw"])
        pitch = float(row["pitch"])
        roll = float(row["roll"])

        exif_dict["GPS"][piexif.GPSIFD.GPSImgDirectionRef] = "T"
        exif_dict["GPS"][piexif.GPSIFD.GPSImgDirection] = (int(yaw * 100), 100)

        # JSON metadata
        extra_cols = [
            "gyro_x","gyro_y","gyro_z",
            "accel_x","accel_y","accel_z",
            "spd_3d","spd_2d","temp"
        ]

        meta_json = {c: row[c] for c in extra_cols}
        meta_json.update({"yaw_deg": yaw, "pitch_deg": pitch, "roll_deg": roll})

        exif_dict["0th"][piexif.ImageIFD.ImageDescription] = json.dumps(meta_json).encode()

        img.save(dst_path, exif=piexif.dump(exif_dict))

        args = [
            "exiftool", "-overwrite_original",
            f"-XMP:PoseYawDegrees={yaw}",
            f"-XMP:PosePitchDegrees={pitch}",
            f"-XMP:PoseRollDegrees={roll}",
            dst_path
        ]
        run(args)

        print(f"{video_id} — {img_name} → OK")

    print(f"\nSELESAI EXIF: {video_id}\n")

    # ============================================================
    # =============== PART 3 — AUTO SPLIT FILE ===================
    # ============================================================

    print("\n=== PART 3: SPLIT KE FOLDER BERDASARKAN INDEX ===")

    base_dir = OUT_IMG_DIR  # gunakan folder output dari video ini

    for filename in os.listdir(base_dir):
        if "_" in filename:
            try:
                index = filename.split("_")[-1].replace(".jpg", "")
                folder_name = f"folder_{index}"
                folder_path = os.path.join(base_dir, folder_name)
                os.makedirs(folder_path, exist_ok=True)

                src = os.path.join(base_dir, filename)
                dst = os.path.join(folder_path, filename)
                shutil.move(src, dst)

                print(f"Moved {filename} -> {folder_name}")

            except Exception as e:
                print(f"Skipping {filename}: {e}")

    print(f" SPLIT SELESAI untuk {video_id}\n")

print("\n==================== ALL VIDEOS DONE ====================\n")



=== LOAD VIDEO DURATIONS FROM TXT ===
Durasi ditemukan untuk 2 video

=== FOUND VIDEO FOLDERS ===
 - GS014244
 - GS014287


======= MEMPROSES VIDEO: GS014244 =======
Durasi video: 59.86 detik
CSV metadata ditemukan: D:\metadata\360\GAGAL\1.360_raw\output\GS014244.csv

=== PART 1: GENERATE CSV MAPPING ===
jumlah grup frame:180
jumlah metadata row: 1089
jumlah metadata tidak sama
yang di proses hanya: 180
sisa frame: 0
sisa_metadata : 909
CSV mapping selesai → D:\metadata\360\GAGAL\5.camera_geotag\GS014244_mapping.csv

=== PART 2: EMBED EXIF + XMP ===
GS014244 — GS014244_image_0001_0.jpg → OK
GS014244 — GS014244_image_0001_1.jpg → OK
GS014244 — GS014244_image_0001_2.jpg → OK
GS014244 — GS014244_image_0001_3.jpg → OK
GS014244 — GS014244_image_0001_4.jpg → OK
GS014244 — GS014244_image_0001_5.jpg → OK
GS014244 — GS014244_image_0001_6.jpg → OK
GS014244 — GS014244_image_0001_7.jpg → OK
GS014244 — GS014244_image_0002_0.jpg → OK
GS014244 — GS014244_image_0002_1.jpg → OK
GS014244 — GS014244_ima